[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AI-Hypercomputer/maxtext/blob/main/src/maxtext/examples/dpo_qwen3_demo.ipynb)

# Qwen3 Preference Optimization (DPO/ORPO) Demo

This notebook demonstrates how to perform Direct Preference Optimization (DPO) and Odds Ratio Preference Optimization (ORPO) on Qwen3-0.6B using the `argilla/distilabel-intel-orca-dpo-pairs` dataset with MaxText and Tunix integration.

## Overview

Direct Preference Optimization (DPO) is a stable and efficient alternative to RLHF (Reinforcement Learning from Human Feedback) that optimizes language models directly on preference pairs without requiring a separate reward model. 

In this notebook, we will:
1.  Set up the environment and install dependencies.
2.  Load a pre-trained Qwen3-0.6B model configuration.
3.  Fine-tune it using DPO on a preference dataset.
4.  Save the resulting model checkpoints.

## Prerequisites

### Get Your Hugging Face Token

To access model checkpoints from the Hugging Face Hub, you need to authenticate with a personal access token.

1.  **Navigate to the Access Tokens page** in your Hugging Face account settings: [https://huggingface.co/settings/tokens](https://huggingface.co/settings/tokens)
2.  **Create a new token** with a `read` role.
3.  **Copy the generated token**.

In [ ]:
try:
  from google.colab import userdata
  print("Running the notebook on Google Colab")
  IN_COLAB = True
except ImportError:
    print("Running the notebook on Visual Studio or JupyterLab")
    IN_COLAB = False

## Installation: MaxText and Post-training Dependencies

In [ ]:
if IN_COLAB:
    # Clone the MaxText repository
    !git clone https://github.com/AI-Hypercomputer/maxtext.git
    %cd maxtext

    # Install uv, a fast Python package installer
    !pip install uv
    
    # Install MaxText and post-training dependencies
    !uv pip install -e .[tpu-post-train] --resolution=lowest
    !install_tpu_post_train_extra_deps
    
    print("\nIMPORTANT: Restart your session (Runtime > Restart session) and run the notebook from the 'Environment Setup' section.")

## Environment Setup

In [ ]:
import datetime
import jax
import os
from maxtext.configs import pyconfig
from maxtext.utils.globals import MAXTEXT_PKG_DIR
from maxtext.trainers.post_train.dpo import train_dpo
from huggingface_hub import login

print(f"MaxText installation path: {MAXTEXT_PKG_DIR}")
print(f"JAX version: {jax.__version__}")
print(f"JAX devices: {jax.devices()}")

In [ ]:
if IN_COLAB:
    from huggingface_hub import notebook_login
    notebook_login()
else:
    from huggingface_hub import login
    login()

## Configuration

We will set up the configuration for Direct Preference Optimization (DPO) training.

In [ ]:
# Model and Experiment details
ALGORITHM = 'dpo' # @param ['dpo', 'orpo']
MODEL_NAME = "qwen3-0.6b" # @param ["qwen3-0.6b", "gemma2-2b", "llama3-8b"]
TOKENIZER_PATH = "Qwen/Qwen3-0.6B" # @param {type:"string"}
STEPS = 20 # @param {type:"integer"}
BASE_OUTPUT_DIRECTORY = f"{MAXTEXT_PKG_DIR}/align_qwen06_output"
RUN_NAME = f"{ALGORITHM}-demo-{datetime.datetime.now().strftime('%Y%m%d-%H%M%S')}"

# Construct configuration arguments
config_argv = [
    "",
    f"{MAXTEXT_PKG_DIR}/configs/post_train/dpo.yml",
    f"run_name={RUN_NAME}",
    f"base_output_directory={BASE_OUTPUT_DIRECTORY}",
    f"model_name={MODEL_NAME}",
    f"tokenizer_path={TOKENIZER_PATH}",
    "dataset_type=hf",
    "hf_path=argilla/distilabel-intel-orca-dpo-pairs",
    "hf_eval_split=train",
    "train_split=train",
    f"steps={STEPS}",
    "per_device_batch_size=1",
    "max_target_length=1024",
    "eval_interval=10",
    "enable_checkpointing=False",
    "log_config=False",
    "use_dpo=True",
    f"dpo.algo='{ALGORITHM}'",
    "enable_goodput_recording=False",
    "monitor_goodput=False",
    "profiler=xplane",
]

config = pyconfig.initialize(config_argv)

print(f"\n✓ {ALGORITHM.upper()} configuration loaded for {MODEL_NAME}")
print(f"  Run name: {config.run_name}")
print(f"  Output directory: {config.base_output_directory}/{config.run_name}")

## Data Preview

It is always a good practice to inspect the dataset before training. DPO requires a dataset with preference pairs (prompt, chosen, rejected).

In [ ]:
from datasets import load_dataset

dataset_name = "argilla/distilabel-intel-orca-dpo-pairs"
print(f"Loading a few samples from {dataset_name}...")
preview_ds = load_dataset(dataset_name, split="train", streaming=True)
samples = list(preview_ds.take(2))

for i, sample in enumerate(samples):
    print(f"\n--- Sample {i+1} ---")
    print(f"PROMPT:\n{sample['input'][:300]}...")
    print(f"\nCHOSEN:\n{sample['chosen'][:300]}...")
    print(f"\nREJECTED:\n{sample['rejected'][:300]}...")

## Run Direct Preference Optimization Training

In [ ]:
print("Starting DPO Training...")
try:
    trainer, mesh = train_dpo.train(config)
    print("DPO Training Complete!")
    print(f"Checkpoints saved to: {config.checkpoint_dir}")
except Exception as e:
    print("Training Failed!")
    print(f"Error: {str(e)}")
    import traceback
    traceback.print_exc()

## 📚 Learn More

- **DPO Paper**: [Direct Preference Optimization: Your Language Model is Secretly a Reward Model](https://arxiv.org/abs/2305.18290)
- **MaxText Documentation**: https://maxtext.readthedocs.io/
- **Tunix Documentation**: https://github.com/google/tunix